In [1]:
# Install dependencies if you haven't already:
# Uninstall the CPU versions to prevent conflicts
# %pip uninstall -y torch torchvision torchaudio

# Install the GPU (CUDA) versions directly from PyTorch
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Install the rest of the required libraries
# %pip install ultralytics opencv-python scikit-learn matplotlib numpy

import cv2
import glob
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 1. Configuration
DATA_DIR = "data/images" # Ensure you have data/images/squat_standing and data/images/squat_bottom
TEST_VIDEO_PATH = "test_video.mp4" # Update with your test video
OUTPUT_VIDEO_PATH = "rep_count_output.mp4"
POSE_CONFIDENCE = 0.5

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load YOLOv8 Nano Pose (downloads automatically if missing)
print("Loading YOLOv8 Pose model...")
yolo_pose_model = YOLO('yolov8n-pose.pt')

Using device: cuda
Loading YOLOv8 Pose model...


In [11]:
def process_image_dataset(data_dir, yolo_model):
    #  Your_Project_Folder/              
    # ├── Rep_Count.ipynb      
    # ├── test_video.mp4                
    # └── data/
    #   └── images/
    #       ├── squat_standing/       
    #       └── squat_bottom/          
    X_data = []
    y_data = []
    classes = ['jump_bottom', 'jump_standing']
    
    for class_name in classes:
        # Check for multiple common image extensions
        image_paths = glob.glob(f"{data_dir}/{class_name}/*.*")
        print(f"Found {len(image_paths)} images for {class_name}")
        
        for img_path in image_paths:
            frame = cv2.imread(img_path)
            if frame is None: 
                continue
                
            # Run YOLO inference
            results = yolo_model(frame, verbose=False)
            
            if len(results) > 0 and results[0].keypoints is not None:
                kpts_data = results[0].keypoints.data
                if len(kpts_data) > 0:
                    kpts = kpts_data[0].cpu().numpy() # Shape: (17, 3)
                    
                    # Only use high-confidence detections
                    if np.mean(kpts[:, 2]) >= POSE_CONFIDENCE:
                        # Normalize coordinates so the model is resolution-independent
                        h, w = frame.shape[:2]
                        kpts_normalized = kpts.copy()
                        kpts_normalized[:, 0] /= w
                        kpts_normalized[:, 1] /= h
                        
                        X_data.append(kpts_normalized.flatten())
                        y_data.append(class_name)
                        
    return np.array(X_data, dtype=np.float32), np.array(y_data)

# Process the dataset
print("Extracting keypoints from images...")
X_raw, y_raw = process_image_dataset(DATA_DIR, yolo_pose_model)

if len(X_raw) == 0:
    raise ValueError("No valid keypoints found. Check your image paths and dataset folders.")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)

# Split into Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X_raw, y_encoded, test_size=0.2, random_state=42)

# Convert to PyTorch Dataloaders
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Dataset ready: {len(X_train)} train samples, {len(X_val)} validation samples.")

Extracting keypoints from images...
Found 51 images for jump_bottom
Found 43 images for jump_standing
Dataset ready: 75 train samples, 19 validation samples.


In [3]:
%pip install tensorflow==2.16.1 tf-keras

%pip install ml-dtypes==0.3.2


^C
Note: you may need to restart the kernel to use updated packages.
  Using cached ml_dtypes-0.3.2-cp312-cp312-win_amd64.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp312-cp312-win_amd64.whl (128 kB)
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.5.4
    Uninstalling ml_dtypes-0.5.4:
      Successfully uninstalled ml_dtypes-0.5.4
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.21.0 requires ml_dtypes>=0.5.0; platform_machine != "s390x", but you have ml-dtypes 0.3.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached ml_dtypes-0.3.2-cp312-cp312-win_amd64.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp312-cp312-win_amd64.whl (128 kB)

  Attempting uninstall: protobuf

    Found existing installation: protobuf 7.34.1

    Uninstalling protobuf-7.34.1:

      Successfully uninstalled protobuf-7.34.1

   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
   ---------------------------------------- 0/2 [protobuf]
  Attempting uninstall: ml-dtypes
   ---------------------------------------- 0/2 [protobuf]
   -------------------- ------------------- 1/2 [ml-dtypes]
    Found existing installation: ml-dtypes 0.3.2
   -------------------- ------------------- 1/2 [ml-dtypes]
    Uninstalling ml

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.21.0 requires ml_dtypes>=0.5.0; platform_machine != "s390x", but you have ml-dtypes 0.3.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tf_keras as keras
import numpy as np

print("Step 1: Building the 'Nuclear Option' Model...")
keras_model = keras.Sequential([
    keras.layers.Input(batch_shape=(1, 51)),
    keras.layers.Dense(128),
    keras.layers.ReLU(max_value=6.0),   # Hard-caps spikes to prevent scale crushing
    keras.layers.Dense(64),
    keras.layers.ReLU(max_value=6.0),   # Hard-caps spikes
    keras.layers.Dense(2, activation = "softmax")
])

keras_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("Step 2: Training...")
X_train_np = np.array(X_train, dtype=np.float32)
y_train_np = np.array(y_train, dtype=np.int32)
keras_model.fit(X_train_np, y_train_np, epochs=30, batch_size=32, validation_split=0.1)

print("Step 3: Quantizing Native Model...")
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    yield [np.zeros((1, 51), dtype=np.float32)]
    yield [np.ones((1, 51), dtype=np.float32)]
    for i in range(len(X_train_np)):
        yield [X_train_np[i].reshape(1, 51)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_path = "rep_counter_int8.tflite"
with open(tflite_path, "wb") as f:
    f.write(converter.convert())

print(f"Saved bulletproof INT8 model to {tflite_path}")

Step 1: Building the 'Nuclear Option' Model...
Step 2: Training...
Epoch 1/30
3/3 [==============================] - 1s 128ms/step - loss: 0.7067 - accuracy: 0.4627 - val_loss: 0.7858 - val_accuracy: 0.2500
Epoch 2/30
3/3 [==============================] - 0s 19ms/step - loss: 0.6870 - accuracy: 0.5373 - val_loss: 0.8272 - val_accuracy: 0.2500
Epoch 3/30
3/3 [==============================] - 0s 19ms/step - loss: 0.6510 - accuracy: 0.5373 - val_loss: 0.7066 - val_accuracy: 0.2500
Epoch 4/30
3/3 [==============================] - 0s 19ms/step - loss: 0.6056 - accuracy: 0.5522 - val_loss: 0.6478 - val_accuracy: 0.7500
Epoch 5/30
3/3 [==============================] - 0s 16ms/step - loss: 0.5860 - accuracy: 0.9701 - val_loss: 0.6053 - val_accuracy: 1.0000
Epoch 6/30
3/3 [==============================] - 0s 19ms/step - loss: 0.5644 - accuracy: 1.0000 - val_loss: 0.5391 - val_accuracy: 1.0000
Epoch 7/30
3/3 [==============================] - 0s 15ms/step - loss: 0.5420 - accuracy: 0.9403 -

INFO:tensorflow:Assets written to: C:\Users\fanju\AppData\Local\Temp\tmpuxjkt9xf\assets
c:\Users\fanju\miniforge3\Lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved bulletproof INT8 model to rep_counter_int8.tflite


In [ ]:
# Save the quantized model to a file
tflite_model_path = "rep_counter_int8.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model_int8)

print(f"Successfully saved quantized model to: {tflite_model_path}")